<a href="https://colab.research.google.com/github/saraxfl/Notebook/blob/main/Collab6_Bloque2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns, sklearn, torch
from torch import nn
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

torch.set_num_threads(1)  # same results run after run on CPU
torch.use_deterministic_algorithms(True)
plt.rcParams['figure.figsize'] = (10, 4)

print('torch', torch.__version__, '| sklearn', sklearn.__version__)


torch 2.11.0+cpu | sklearn 1.6.1


In [2]:

flights = sns.load_dataset('flights')  # 144 rows: year, month, passengers (Jan 1949 - Dec 1960)

y = pd.Series(
    flights['passengers'].values,
    index=pd.date_range('1949-01-01', periods=144, freq='MS'),
    name='passengers'
)

assert len(y) == 144 and y.sum() == 40363

train, test = y[:'1958'], y['1959':]

H = len(test)  # 24 months ahead, all forecast in December 1958

def rmse(f, a):
    return float(np.sqrt(np.mean((np.asarray(f, float) - np.asarray(a, float)) ** 2)))

def mape(f, a):
    return float(100 * np.mean(np.abs((np.asarray(f, float) - np.asarray(a, float)) / np.asarray(a, float))))

results = {}

snaive = np.tile(train.values[-12:], 2)  # 1958 copied into 1959 and 1960

results['Seasonal naive'] = (
    rmse(snaive, test),
    mape(snaive, test)
)


In [3]:


def make_windows(s, p):
    s = np.asarray(s, float)
    X = np.array([s[i - p:i] for i in range(p, len(s))])
    return X, s[p:]


In [5]:

def recursive_forecast(model, history, p, h):
    hist = list(np.asarray(history, float))
    out = []
    for _ in range(h):
        nxt = float(model.predict(np.array(hist[-p:]).reshape(1, -1))[0])
        out.append(nxt)
        hist.append(nxt)
    return np.array(out)


def z_of(Lser):
    Lser = np.asarray(Lser, float)
    return Lser[13:] - Lser[12:-1] - Lser[1:-12] + Lser[:-13]


def undo(z_future, L_hist):
    Lh = list(L_hist)
    for zt in z_future:
        Lh.append(zt + Lh[-1] + Lh[-12] - Lh[-13])
    return np.array(Lh[len(L_hist):])


In [6]:

p = 13
L = np.log(y.values)
Ltr = L[:len(train)]

ztr = z_of(Ltr)

mz, sz = ztr.mean(), ztr.std()

Xz, tz = make_windows((ztr - mz) / sz, p)

f_zmean = np.exp(undo(np.full(H, mz), Ltr))

results['Transform, no learning'] = (
    rmse(f_zmean, test),
    mape(f_zmean, test)
)


lin = LinearRegression().fit(Xz, tz)
f_lin = np.exp(undo(
    recursive_forecast(lin, (ztr - mz) / sz, p, H) * sz + mz,
    Ltr
))
results['Linear AR(13)'] = (rmse(f_lin, test), mape(f_lin, test))

for k, (r, m) in results.items():
    print(f'{k:31s} RMSE {r:6.1f}   MAPE {m:5.1f}%')

assert abs(results['Seasonal naive'][0] - 77.0) < 0.05 and abs(results['Linear AR(13)'][0] - 67.6) < 0.05



Seasonal naive                  RMSE   77.0   MAPE  15.5%
Transform, no learning          RMSE   76.8   MAPE  15.4%
Linear AR(13)                   RMSE   67.6   MAPE  13.2%


In [7]:
def to_seq(X):                                    # (n, 13) table  ->  (n, 13, 1) tensor of sequences
    return torch.tensor(np.asarray(X, float), dtype=torch.float32).unsqueeze(-1)

def to_target(t):
    return torch.tensor(np.asarray(t, float), dtype=torch.float32)

print('table:', Xz.shape, '->  sequences:', tuple(to_seq(Xz).shape))

table: (94, 13) ->  sequences: (94, 13, 1)
